# Runbook 5.6.c: Digital evidence analysis and attack classification

**Skill:** Detect Attacks, Collaborate · **Maps to:** 5.6.B, 5.6.C, 5.6.D, 5.6.E · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Formalize the analyst reasoning process from the lesson: given a bundle of evidence indicators
collected during an investigation, score each known attack category by how many of its
indicators are present, and return the best-supported classification.

**Lesson objective.** Explain the chain-of-custody principles that make digital evidence trustworthy, describe the difference between an indicator of compromise (IOC) and a conclusion about attack type, and classify a set of collected evidence into the attack category it most strongly supports.

### What comes in
`indicators`: `list[str]` of evidence tags (may contain duplicates).

### What must come out
`str`, the category with the most matching indicators (as a set; duplicates don't add extra
weight). Ties go to the alphabetically first tied category. `"unknown"` if nothing matches.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
["multiple_401_responses", "single_source_ip"]  ->  "brute-force"
["server_restarted"]                             ->  "unknown"
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(indicators: list) -> str
```

#### Input (`indicators`)
`list[str]`, evidence tags collected during an investigation. May contain duplicates (treated
as a set, duplicates do not increase a category's score).

#### Known indicator-to-category mapping
- `"brute-force"`: `multiple_401_responses`, `single_source_ip`, `many_usernames_targeted`,
  `credential_guessing_pattern`
- `"sql-injection"`: `sql_meta_characters_in_request`, `database_error_in_response`,
  `union_select_pattern`, `unexpected_query_result_size`
- `"malware"`: `known_malware_file_hash`, `unexpected_process_spawned`,
  `outbound_c2_connection`, `persistence_mechanism_installed`
- `"phishing"`: `suspicious_email_attachment`, `credential_harvesting_url`,
  `spoofed_sender_domain`, `user_reported_suspicious_email`

#### Output (`result`)
`str`, the category name whose known indicators appear most often (as a set) in `indicators`.
Ties are broken by choosing the alphabetically first category among the tied leaders. If no
indicator in the input matches any known category, return `"unknown"`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Indicators outside the known mapping are ignored (they contribute to no category's score).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 5.6.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List


def solve(indicators: List[str]) -> str:
    """Classify an evidence bundle into the attack category its indicators most strongly
    support (highest count of matching known indicators for that category).

    Args:
        indicators: list[str] of evidence tags collected during an investigation.
    Returns:
        str — the best-supported category name ("brute-force", "sql-injection", "malware",
        "phishing"), or "unknown" if no indicator matches any known category. Ties are broken
        by choosing the alphabetically first category among the tied leaders. See
        the Reference section below for the exact indicator-to-category mapping.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_clear_brute_force():
    indicators = ["multiple_401_responses", "single_source_ip", "many_usernames_targeted"]
    assert impl.solve(indicators) == "brute-force"


def test_clear_sql_injection():
    indicators = ["sql_meta_characters_in_request", "database_error_in_response"]
    assert impl.solve(indicators) == "sql-injection"


def test_clear_malware():
    indicators = ["known_malware_file_hash", "outbound_c2_connection", "persistence_mechanism_installed"]
    assert impl.solve(indicators) == "malware"


def test_clear_phishing():
    indicators = ["suspicious_email_attachment", "credential_harvesting_url", "spoofed_sender_domain"]
    assert impl.solve(indicators) == "phishing"


def test_no_matching_indicators_unknown():
    indicators = ["server_restarted", "disk_space_low"]
    assert impl.solve(indicators) == "unknown"


def test_mixed_weak_signals_highest_score_wins():
    indicators = [
        "multiple_401_responses",
        "sql_meta_characters_in_request",
        "database_error_in_response",
        "union_select_pattern",
    ]
    assert impl.solve(indicators) == "sql-injection"


def test_tie_broken_alphabetically():
    indicators = ["multiple_401_responses", "known_malware_file_hash"]
    assert impl.solve(indicators) == "brute-force"


def test_empty_indicators_unknown():
    assert impl.solve([]) == "unknown"


def test_unrelated_indicators_ignored():
    indicators = ["unexpected_process_spawned", "server_restarted", "disk_space_low"]
    assert impl.solve(indicators) == "malware"


def test_duplicate_indicators_not_double_counted():
    indicators = ["multiple_401_responses", "multiple_401_responses", "known_malware_file_hash"]
    assert impl.solve(indicators) == "brute-force"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve(["multiple_401_responses", "single_source_ip"]),
     "brute-force"),
    (lambda: solve(["server_restarted"]),
     "unknown"),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it